<a href="https://colab.research.google.com/github/fandri-indranata/data-science-2026/blob/main/Pertemuan3_FandriIndranata_230401010180.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import library yang diperlukan
import pandas as pd
import numpy as np
import requests
from scipy.stats.mstats import winsorize

# URL Dataset (Link Google Drive yang sudah dikonversi agar bisa dibaca pd.read_csv)
url_dataset = "https://drive.google.com/uc?export=download&id=1LfQWProB0VjWN5q8bKuRIgn-stULfIRo"

# STEP 0: Load & eksplorasi awal
df = pd.read_csv(url_dataset)
print("=== Shape Awal ===")
print(df.shape)
print("\n=== Jumlah Missing Values Awal ===")
print(df.isnull().sum())
print("\n=== Informasi Data ===")
print(df.info())

In [17]:
# STEP 1: Hapus Duplikat (M2)
df.drop_duplicates(inplace=True)
print("Shape setelah hapus duplikat:", df.shape)

# STEP 2: Normalisasi String (M4)
# Menghilangkan spasi di awal/akhir dan mengubah format huruf
df['kota'] = df['kota'].str.strip().str.title()      # Contoh: 'jakarta' -> 'Jakarta'
df['kondisi'] = df['kondisi'].str.strip().str.lower() # Contoh: 'Baik' -> 'baik'

# STEP 3: Imputasi Missing Values (M1)
# Numerik: gunakan Median (lebih robust terhadap outlier)
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())
# Kategorik: gunakan Modus (nilai yang paling sering muncul)
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

# STEP 4: Tangani Outlier dengan IQR Fence / Capping (M3)
# Kita terapkan pada kolom numerik yang berpotensi outlier
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    # Capping (Winsorization): nilai di luar batas diganti dengan nilai batas
    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

# STEP 5: Validasi Akhir
print("\n=== Validasi Akhir ===")
print("Total Missing Values:", df.isnull().sum().sum())
print("Total Duplikat:", df.duplicated().sum())
print("Shape Akhir:", df.shape)

# STEP 6: Ekspor Dataset Bersih
df.to_csv('housing_clean.csv', index=False)
print("Dataset bersih berhasil disimpan sebagai 'housing_clean.csv'")

Shape setelah hapus duplikat: (130, 7)

=== Validasi Akhir ===
Total Missing Values: 0
Total Duplikat: 0
Shape Akhir: (130, 7)
Dataset bersih berhasil disimpan sebagai 'housing_clean.csv'


In [ ]:
# STEP 7: Akses REST API Publik (JSONPlaceholder)
url_api = "https://jsonplaceholder.typicode.com/users"

try:
    # Mengirim request GET dengan timeout 10 detik
    response = requests.get(url_api, timeout=10)

    # Cek apakah request berhasil (Status Code 200)
    if response.status_code == 200:
        data_api = response.json() # Ubah respons menjadi format JSON/Dictionary

        # Konversi ke DataFrame.
        # Karena ada nested JSON (misal: address.city), maka digunakan json_normalize
        df_api = pd.json_normalize(data_api, sep='_')

        # Tampilkan beberapa kolom yang relevan
        print("=== Data dari API Berhasil Dimuat ===")
        display(df_api[['id', 'name', 'email', 'address_city']].head())
    else:
        print(f"Gagal mengambil data. Status Code: {response.status_code}")

except requests.exceptions.RequestException as e:
    print(f"Terjadi kesalahan koneksi: {e}")